Here, I will combine all Pando eGRN objects into a single eGRN object.

=========================================================

Important info about how Pando selects the features (target genes) out of the\
user-provided input genes to infer eGRNs:

Firstly, Pando takes a seurat object with RNA and peak assays as input.\
In addition, it takes "genes" argument which contains the gene names, so\
called features to fit models against. 

Importantly, Pando does two things to determine features. It filters in\
inputted features that are included in the RNA assay. Out of those filtered features\
it also filters those genes whose information is available in the "Annotation"\
object that is stored in the ChromatinAssay object containing the peaks.\
Then, it selects those features (target genes) that satisfy both of these\
criteria and starts to combine their respective peak and gene expression\
information in the Seurat object and fits the model per each of the selected\
feature.

For that reason, the number of input genes and the number of the genes\
present in the final model is not equal for most of the time.

==========================================================

Although I divided my target genes into 500 genes of chunks with overlapping\
5 genes in-between consecutive chunks, Pando will out-select some of them\
and so it is most likely that at least some of the consecutive chunks will\
not contain overlapping genes.

It is better to identify the set of genes out of all genes that Pando\
can use for modelling before starting the inference of eGRNs, so that\
one makes sure that all chunks contain overlapping sets to verify that\
no chunk-relevant variation exists between the fitted models.
HOWEVER, I checked before in my analysis with Zhu et al. 2023 data\
confirmed that independent of chunks, the fitted models for the same\
are almost identical in terms number of variable and the sign and magnitude\
of the coefficients.

==========================================================

Below, I will check the number of features (target genes) present in\
each chunk objects and the number of overlapping genes between them\
and then combine these objects into a single Pando eGRN object.

==========================================================================

In [1]:
getwd()

[1] "/fast/AG_Bunina/Yusuf/Project_Endothelial_and_Stroke/Datasets/Chromatin_and_Gene_Exp/2024_C_A_Mannens_C_et_al/04_02_25"

In [2]:
here::here()

[1] "/fast/AG_Bunina/Yusuf/Project_Endothelial_and_Stroke/Datasets/Chromatin_and_Gene_Exp/2024_C_A_Mannens_C_et_al/04_02_25"

In [3]:
# load the R environment with the necessary packages such as Epiregulon:

my_epiregulon_lib <- here::here("renv", "library/linux-rhel-9.4/R-4.4/x86_64-unknown-linux-gnu")

In [4]:
.libPaths(new = my_epiregulon_lib, include.site = FALSE)

In [5]:
.libPaths()

[1] "/fast/AG_Bunina/Yusuf/Project_Endothelial_and_Stroke/Datasets/Chromatin_and_Gene_Exp/2024_C_A_Mannens_C_et_al/04_02_25/renv/library/linux-rhel-9.4/R-4.4/x86_64-unknown-linux-gnu"
[2] "/gnu/store/29x2k7i71g9xq09xmbj1lk515cl7if63-r-minimal-4.4.2/lib/R/library"

In [6]:
library(magrittr)

In [7]:
here::here('r_objects') |> list.dirs() %>% basename()

[1] "r_objects"               ".ipynb_checkpoints"     
[3] "TF_activity_chunks"      "pando_eGRN_chunks_final"

In [8]:
# view eGRN objects in windows-style order:

here::here('r_objects', 'pando_eGRN_chunks_final') |> list.files() %>% gtools::mixedsort() %>% print()

 [1] "mannens_et_al_w_eGRNs_SLURM_c1.RDS"  "mannens_et_al_w_eGRNs_SLURM_c2.RDS" 
 [3] "mannens_et_al_w_eGRNs_SLURM_c3.RDS"  "mannens_et_al_w_eGRNs_SLURM_c4.RDS" 
 [5] "mannens_et_al_w_eGRNs_SLURM_c5.RDS"  "mannens_et_al_w_eGRNs_SLURM_c6.RDS" 
 [7] "mannens_et_al_w_eGRNs_SLURM_c7.RDS"  "mannens_et_al_w_eGRNs_SLURM_c8.RDS" 
 [9] "mannens_et_al_w_eGRNs_SLURM_c9.RDS"  "mannens_et_al_w_eGRNs_SLURM_c10.RDS"
[11] "mannens_et_al_w_eGRNs_SLURM_c11.RDS" "mannens_et_al_w_eGRNs_SLURM_c12.RDS"
[13] "mannens_et_al_w_eGRNs_SLURM_c13.RDS" "mannens_et_al_w_eGRNs_SLURM_c14.RDS"
[15] "mannens_et_al_w_eGRNs_SLURM_c15.RDS" "mannens_et_al_w_eGRNs_SLURM_c16.RDS"
[17] "mannens_et_al_w_eGRNs_SLURM_c17.RDS" "mannens_et_al_w_eGRNs_SLURM_c18.RDS"
[19] "mannens_et_al_w_eGRNs_SLURM_c19.RDS" "mannens_et_al_w_eGRNs_SLURM_c20.RDS"
[21] "mannens_et_al_w_eGRNs_SLURM_c21.RDS" "mannens_et_al_w_eGRNs_SLURM_c22.RDS"
[23] "mannens_et_al_w_eGRNs_SLURM_c23.RDS" "mannens_et_al_w_eGRNs_SLURM_c24.RDS"
[25] "mannens_et_al_w_eGRNs_

In [9]:
library(tidyverse)

-- Attaching core tidyverse packages ------------------------ tidyverse 2.0.0 --
v dplyr     1.1.4     v readr     2.1.5
v forcats   1.0.0     v stringr   1.5.1
v ggplot2   3.5.1     v tibble    3.2.1
v lubridate 1.9.3     v tidyr     1.3.1
v purrr     1.0.4     
-- Conflicts ------------------------------------------ tidyverse_conflicts() --
x tidyr::extract()   masks magrittr::extract()
x dplyr::filter()    masks stats::filter()
x dplyr::lag()       masks stats::lag()
x purrr::set_names() masks magrittr::set_names()
i Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


In [10]:
library(Seurat)

Loading required package: SeuratObject

Loading required package: sp

'SeuratObject' was built under R 4.4.1 but the current version is
4.4.2; it is recomended that you reinstall 'SeuratObject' as the ABI
for R may have changed


Attaching package: 'SeuratObject'


The following objects are masked from 'package:base':

    intersect, t




In [11]:
library(Signac)

In [12]:
library(Pando)


Attaching package: 'Pando'


The following objects are masked from 'package:Seurat':

    GetAssay, VariableFeatures


The following objects are masked from 'package:SeuratObject':

    LayerData, VariableFeatures




In [26]:
# To view how Pando extracts gene annotations and determines target genes for modelling,
# I will capture the relevant Pando function script and search some patterns of 'annotation':

Pando:::fit_grn_models.GRNData %>% capture.output() %>% grep(x = . , pattern = 'Annotation', value = TRUE)

[1] "    gene_annot <- Signac::Annotation(GetAssay(object, params$peak_assay))"

In [28]:
# search for 'gene_annot'

Pando:::fit_grn_models.GRNData %>% print() %>% capture.output() %>% grep(x = . , pattern = 'gene_annot', value = TRUE)

[1] "    gene_annot <- Signac::Annotation(GetAssay(object, params$peak_assay))"                      
[2] "    if (is.null(gene_annot)) {"                                                                 
[3] "    features <- intersect(gene_annot$gene_name, genes) %>% intersect(rownames(GetAssay(object, "
[4] "    gene_annot <- gene_annot[gene_annot$gene_name %in% features, "                              
[5] "            method = peak_to_gene_method, genes = gene_annot, "

In [30]:
# Inport first eGRN object to access Seurat object:

mannens_et_al_w_eGRNs_SLURM_c1 <- 
    readRDS(here::here('r_objects', 'pando_eGRN_chunks_final', 'mannens_et_al_w_eGRNs_SLURM_c1.RDS'))

In [31]:
# Retrieve annotations from the Chromatin Assay as done by Pando:

gene_annot <- Signac::Annotation(object = mannens_et_al_w_eGRNs_SLURM_c1@data@assays$peaks)

In [32]:
mannens_et_al_w_eGRNs_SLURM_c1

An object of class "GRNData"
Slot "grn":
A RegulatoryNetwork object based on 1122 transcription factors

1 inferred network: glm_network

Slot "data":
An object of class Seurat 
430192 features across 49470 samples within 2 assays 
Active assay: RNA (25071 features, 5000 variable features)
 3 layers present: counts, data, scale.data
 1 other assay present: peaks


In [36]:
# Calculate the number of genes in the RNA assay that are available in the annotation object:

intersect(gene_annot$gene_name, rownames(mannens_et_al_w_eGRNs_SLURM_c1@data)) %>% length()

[1] 19042

In [37]:
# total genes:

rownames(mannens_et_al_w_eGRNs_SLURM_c1@data) %>% length()

[1] 25071

In [38]:
# So 19042 genes out of 25071 could be fitted model against.

In [39]:
# Now, I will use the same function that I used to generate chunks of genes
# to infer eGRNs with Pando. I will retrieve the gene chunks and check
# how many of them are available in the annotation GRanges obkect.

In [41]:
retrieve_in_chunks <- function(gene_names, chunk_size, overlap = 5) {
  
  # Get the total number of gene names provided in the input
  n <- length(gene_names)  

  # Define the step size (how much to move forward after creating each chunk).
  # The step size is calculated by subtracting the overlap from the chunk size.
  # This ensures that each successive chunk shares 'overlap' number of genes with the previous chunk.
  step_size <- chunk_size - overlap  

  # Initialize an empty list to store the resulting chunks of gene names.
  chunk_list <- list()  

  # Use a for-loop to iterate through the gene names vector in steps of 'step_size'.
  # The 'seq()' function generates a sequence of starting points (i) for each chunk.
  # The loop increments by 'step_size' to determine the next chunk’s starting point.
  for (i in seq(1, n, by = step_size)) {
    
    # Extract the current chunk of gene names from the vector.
    # The 'min()' function ensures that the chunk doesn't go beyond the last gene in 'gene_names'.
    # The chunk starts at 'i' and ends at 'i + chunk_size - 1' (or the last available gene).
    chunk <- gene_names[i:min(i + chunk_size - 1, n)]
    
    # Generate a name for each chunk (optional, but helpful for referencing).
    # The 'ceiling()' function calculates which chunk we are in based on 'i' and 'step_size'.
    chunk_name <- paste0("chunk_", ceiling(i / step_size))
    
    # Add the current chunk to the 'chunk_list' under its generated name.
    # This allows for easy access to each chunk by its name.
    chunk_list[[chunk_name]] <- chunk
  }
  
  # Return the full list of chunks. Each element of the list corresponds to a chunk of gene names.
  return(chunk_list)  
}

In [42]:
# retrieve chunks of genes from the Seurat objects store at data slot of Pando object.

gene_chunks <- retrieve_in_chunks(gene_names = rownames(mannens_et_al_w_eGRNs_SLURM_c1@data), 
                                  chunk_size = 500,
                                  overlap = 5)

In [44]:
gene_chunks %>% head()

$chunk_1
  [1] "MALAT1"     "AUTS2"      "NRXN1"      "MIR99AHG"   "GRID2"     
  [6] "NPAS3"      "ERBB4"      "NRXN3"      "NFIA"       "ADGRB3"    
 [11] "PTN"        "DCC"        "NFIB"       "LSAMP"      "TCF4"      
 [16] "PTPRD"      "QKI"        "MAP2"       "ADGRL3"     "PCDH9"     
 [21] "HSP90AA1"   "CADM2"      "SOX4"       "NRG3"       "NKAIN3"    
 [26] "GPM6A"      "DACH1"      "ROBO2"      "TMSB10"     "RORA"      
 [31] "TUBA1A"     "CTNND2"     "CTNNA2"     "ZBTB20"     "CASC15"    
 [36] "NRG1"       "LINC00461"  "XIST"       "SOX5"       "CNTNAP2"   
 [41] "PTPRZ1"     "MAP1B"      "TENM2"      "SYT1"       "RERE"      
 [46] "RPL37A"     "RBFOX1"     "MEIS2"      "ACTB"       "ROBO1"     
 [51] "NLGN1"      "TMSB4X"     "ADGRV1"     "PRKG1"      "PDE4D"     
 [56] "RPLP1"      "RPS11"      "FOXP2"      "CELF2"      "LRRC4C"    
 [61] "ANKS1B"     "FTX"        "MARCKS"     "CSMD1"      "RPL13A"    
 [66] "CCSER1"     "NTM"        "NCKAP5"     "ACTG1"      "NCAM1"     
 [71] "AFF3"       "NAV3"       "PBX1"       "RPL10"      "FRMD4A"    
 [76] "GPC6"       "AKAP6"      "RPL37"      "MAGI2"      "JMJD1C"    
 [81] "GRIK2"      "TCF12"      "HBA2"       "DOCK4"      "SSBP2"     
 [86] "TTC28"      "CACNA2D1"   "RPL13"      "FAM155A"    "CDH2"      
 [91] "NOVA1"      "RFX3"       "SETBP1"     "ANK2"       "ZEB1"      
 [96] "MSI2"       "NAV2"       "SPP1"       "PARD3B"     "ARID1B"    
[101] "NBEA"       "LRP1B"      "DCLK1"      "RPL34"      "MAPK10"    
[106] "DST"        "MAML2"      "WSB1"       "TENM3"      "RPS29"     
[111] "PBX3"       "RPS19"      "THSD7A"     "RPS27A"     "RPS14"     
[116] "RPS18"      "TPT1"       "RPS24"      "AL589740.1" "DLGAP1"    
[121] "KALRN"      "KCNH7"      "CLASP2"     "SOX6"       "ANK3"      
[126] "BAZ2B"      "FBXL7"      "PLXDC2"     "ZFPM2"      "RPLP2"     
[131] "RPS27"      "TRIO"       "RPL23"      "RPS12"      "RUNX1T1"   
[136] "DMD"        "VIM"        "PTMA"       "MACF1"      "MARCHF1"   
[141] "DCLK2"      "CHD7"       "FYN"        "NCAM2"      "MED13L"    
[146] "RPL32"      "UNC5C"      "SLC1A3"     "AGAP1"      "PLEKHA5"   
[151] "PARD3"      "APP"        "SLIT2"      "MLLT3"      "EIF4G3"    
[156] "SEMA6D"     "PLCB1"      "DSCAM"      "PDE4B"      "GPM6B"     
[161] "KIF1B"      "CST3"       "PTPRG"      "HNRNPA2B1"  "LRRTM4"    
[166] "ZNF292"     "MBD5"       "MAGI1"      "RPL11"      "ZSWIM6"    
[171] "EBF1"       "RPS8"       "NTRK2"      "ZEB2"       "RPS2"      
[176] "GPHN"       "CNTN5"      "MAP4K4"     "DLG2"       "RTN4"      
[181] "ZFAND3"     "PPFIA2"     "RPL28"      "FN1"        "MYT1L"     
[186] "C1orf61"    "DPP6"       "FNBP1L"     "SORBS2"     "TMTC2"     
[191] "VCAN"       "ZNF638"     "LRRC7"      "DDX17"      "AFDN"      
[196] "NEDD4L"     "RPL30"      "FTL"        "RPL35A"     "KAZN"      
[201] "RPS6"       "RPL19"      "HMGB1"      "H3-3B"      "WWOX"      
[206] "GNAQ"       "EXOC4"      "CADM1"      "PHIP"       "RPL27A"    
[211] "NF1"        "RPL24"      "GNAS"       "RPS16"      "IGF1R"     
[216] "PTK2"       "KMT2E"      "HBA1"       "TNRC6B"     "PTPRM"     
[221] "KCND2"      "RPS15"      "ARHGAP21"   "SLC8A1"     "SRGAP3"    
[226] "CCDC88A"    "RPS3A"      "RPS20"      "SMYD3"      "SYNE2"     
[231] "MAML3"      "PHACTR1"    "IMMP2L"     "NNAT"       "RPS15A"    
[236] "TNIK"       "KMT2C"      "RPS4X"      "GRIP1"      "TNRC6A"    
[241] "TTC3"       "HSPH1"      "DAB1"       "MEIS1"      "ATRX"      
[246] "RPS28"      "FMNL2"      "AKAP9"      "RPS21"      "DPP10"     
[251] "CHD9"       "NEGR1"      "RPL38"      "RALYL"      "CSMD3"     
[256] "KLF12"      "EFNA5"      "EPHA5"      "EXT1"       "PHF14"     
[261] "NRCAM"      "ARGLU1"     "PCDH7"      "UNC5D"      "FAT3"      
[266] "PPP2R2B"    "SDK1"       "TSC22D1"    "RPL31"      "STMN1"     
[271] "CACHD1"     "BASP1"      "TUBB2B"     "CPE"        "NIPBL"     
[276] "RBMS3"      "WDFY3"      "ZNF804A"    "ZNF385D"    "TEAD1"     
[281] "SPAG9"  

In [49]:
gene_chunks %>% tail()

$chunk_46
  [1] "AC138761.1"   "AL157884.2"   "LINC02383"    "AC104961.1"   "AL133163.3"  
  [6] "AC131254.2"   "AC007106.1"   "APOBEC1"      "AC010420.1"   "AC018630.2"  
 [11] "LGALS7"       "AC011601.1"   "AL109936.3"   "VCX3A"        "TMEM88B"     
 [16] "LINC02033"    "POU5F1B"      "SCGB1D2"      "AL590399.1"   "NPVF"        
 [21] "SLC22A12"     "AP000402.1"   "MIP"          "LINC00545"    "AC016877.1"  
 [26] "AC023347.1"   "AC110767.1"   "AL596442.2"   "MTRNR2L10"    "EVPLL"       
 [31] "LINC01925"    "AC007463.1"   "AL445465.2"   "AC078880.2"   "AC012668.2"  
 [36] "FAM74A6"      "LINC02574"    "AP007216.2"   "LINC01187"    "REG4"        
 [41] "AC005019.2"   "AC018521.7"   "LINC02483"    "AL135910.1"   "AC084116.1"  
 [46] "PCAT14"       "LINC02130"    "LINC01861"    "AC016266.1"   "KRT7"        
 [51] "MMP3"         "SMIM9"        "CALHM1"       "LINC02493"    "AC074344.2"  
 [56] "LINC01741"    "MS4A13"       "LINC02862"    "AL356234.3"   "AC002511.1"  
 [61] "CHRFAM7A"     "AC125603.3"   "LINC00543"    "AC004449.1"   "HOXA9"       
 [66] "KLK8"         "PCAT5"        "LINC00456"    "CELA3A"       "TMEM132D-AS2"
 [71] "LINC00029"    "AC008543.4"   "AC093627.5"   "LINC02423"    "LYZL2"       
 [76] "AL080284.1"   "PYCR3"        "AC093903.1"   "C9orf135-DT"  "AP003351.1"  
 [81] "AL360181.4"   "AL645634.2"   "AC036222.1"   "AL136099.1"   "AC007998.4"  
 [86] "LINC01489"    "AC092634.4"   "AC242842.1"   "AL353614.1"   "AC245033.2"  
 [91] "PGLYRP4"      "AL590483.2"   "AC136188.1"   "ZNF878"       "SLC5A8"      
 [96] "DMRTB1"       "ISX"          "LINC01849"    "LINC02598"    "PRSS33"      
[101] "AC062032.1"   "LINC02031"    "KRT79"        "FAM240B"      "LINC02682"   
[106] "AL157931.2"   "LY6G5C"       "AL109933.1"   "LINC01634"    "AC091043.1"  
[111] "MIR4290HG"    "AC025428.1"   "AC092687.2"   "AL139393.2"   "NOTCH2NLA"   
[116] "LINC01710"    "AL358292.1"   "ZNF705A"      "AC024587.2"   "ZNF707"      
[121] "AC073218.1"   "AC124893.1"   "LINC01733"    "AC092620.2"   "LINC01910"   
[126] "LINC01595"    "SALRNA1"      "AC084768.1"   "LINC01097"    "SOHLH1"      
[131] "AL357143.1"   "AC113608.1"   "KLHL38"       "LINC02020"    "AC012485.2"  
[136] "AC010327.2"   "AC013448.2"   "AP002856.1"   "LINC02420"    "LINC01143"   
[141] "LINC01975"    "CYP4F8"       "DEFB123"      "LINC00239"    "RXFP4"       
[146] "AC012363.2"   "LINC00333"    "AC097501.1"   "CYP4A22"      "ZNF679"      
[151] "CAMP"         "CEACAM16"     "HAND1"        "AC063976.1"   "KRT28"       
[156] "AL109610.1"   "OR1I1"        "LINC01553"    "FAM95B1"      "AL136972.1"  
[161] "LGALS12"      "LILRB3"       "SLC22A18AS"   "AC020656.2"   "CCR4"        
[166] "KRT73"        "CRISP3"       "SSTR3"        "AC145625.1"   "AC034232.1"  
[171] "AC011131.1"   "LINC00974"    "AL591503.1"   "LINC01987"    "ZNF479"      
[176] "ASB17"        "LINC01432"    "UGT1A6"       "LINC02578"    "AC093909.2"  
[181] "TRIML1"       "AL359081.1"   "FKBPL"        "AL450327.1"   "AC099794.1"  
[186] "HSD3B1"       "AF241725.1"   "AC034223.1"   "LINC02255"    "LINC02070"   
[191] "AL807761.3"   "LINC01474"    "LINC00547"    "LINC00474"    "TTLL11-IT1"  
[196] "AC003685.1"   "RFPL4B"       "IL17F"        "AL353740.1"   "TMEM252"     
[201] "OR8B8"        "AC005244.2"   "AC073257.1"   "SERPINA9"     "TNP1"        
[206] "AC241377.3"   "OR10D3"       "AC063952.4"   "LINC02252"    "AC092329.1"  
[211] "OR2D3"        "AC103808.3"   "AC106871.1"   "LINC02573"    "HOXD8"       
[216] "LINC02644"    "NAA80"        "MYOG"         "OR2M2"        "LINC02686"   
[221] "HSD3B2"       "LINC02131"    "AC011509.1"   "AL606970.2"   "AP000477.1"  
[226] "LINC02011"    "AC112187.3"   "UCN2"         "LINC01874"    "AL138731.1"  
[231] "LINC02344"    "AP001790.1"   "AF127577.5"   "AC131212.1"   "AL049536.1"  
[236] "AC104574.1"   "FABP2"        "AC103740.2"   "AL390778.2"   "LINC01193"   
[241] "AC104794.2"   "AC008869.1"   "OR10J1"       "AL139254.1"   "MOB3C"       
[246] "DPEP2NB"      

In [53]:
gene_chunks %>% lapply(length) %>% unlist() %>% print()

 chunk_1  chunk_2  chunk_3  chunk_4  chunk_5  chunk_6  chunk_7  chunk_8 
     500      500      500      500      500      500      500      500 
 chunk_9 chunk_10 chunk_11 chunk_12 chunk_13 chunk_14 chunk_15 chunk_16 
     500      500      500      500      500      500      500      500 
chunk_17 chunk_18 chunk_19 chunk_20 chunk_21 chunk_22 chunk_23 chunk_24 
     500      500      500      500      500      500      500      500 
chunk_25 chunk_26 chunk_27 chunk_28 chunk_29 chunk_30 chunk_31 chunk_32 
     500      500      500      500      500      500      500      500 
chunk_33 chunk_34 chunk_35 chunk_36 chunk_37 chunk_38 chunk_39 chunk_40 
     500      500      500      500      500      500      500      500 
chunk_41 chunk_42 chunk_43 chunk_44 chunk_45 chunk_46 chunk_47 chunk_48 
     500      500      500      500      500      500      500      500 
chunk_49 chunk_50 chunk_51 
     500      500      321 


In [54]:
# Now, find the fragment of those genes that are available in the annotation object: 

target_genes_avaliable <- lapply(gene_chunks, function(x) {

    intersect(gene_annot$gene_name, x)

})

In [55]:
target_genes_avaliable %>% head(1)

$chunk_1
  [1] "NLGN4X"    "TMSB4X"    "GPM6B"     "IL1RAPL1"  "DMD"       "CASK"     
  [7] "RPS4X"     "XIST"      "FTX"       "ATRX"      "RBMX"      "FGF13"    
 [13] "RPL10"     "PLCB1"     "MACROD2"   "CST3"      "RBM39"     "NNAT"     
 [19] "PTPRT"     "GNAS"      "RPS21"     "CAMTA1"    "RERE"      "KIF1B"    
 [25] "KAZN"      "EIF4G3"    "RPL11"     "STMN1"     "SFPQ"      "MACF1"    
 [31] "RPS8"      "DAB1"      "JUN"       "NFIA"      "CACHD1"    "PDE4B"    
 [37] "LRRC7"     "SRSF11"    "NEGR1"     "ADGRL2"    "RPL5"      "FNBP1L"   
 [43] "PTBP2"     "RPS27"     "C1orf61"   "PBX1"      "POU2F1"    "RABGAP1L" 
 [49] "RASAL2"    "KCNT2"     "ENAH"      "CDC42BPA"  "SIPA1L2"   "ARID4B"   
 [55] "HNRNPU"    "SMYD3"     "TUBB2B"    "PHACTR1"   "ATXN1"     "CDKAL1"   
 [61] "SOX4"      "CASC15"    "RPS18"     "RPS10"     "RPL10A"    "ZFAND3"   
 [67] "HSP90AB1"  "SUPT3H"    "DST"       "ADGRB3"    "PHIP"      "ZNF292"   
 [73] "BACH2"     "EPHA7"     "FUT9"      "PNISR"     "GRIK2"     "REV3L"    
 [79] "FYN"       "MARCKS"    "SLC35F1"   "NKAIN2"    "RPS12"     "ARID1B"   
 [85] "QKI"       "AFDN"      "CHL1"      "CNTN4"     "SRGAP3"    "SETD5"    
 [91] "RPL32"     "TBC1D5"    "ZNF385D"   "UBE2E2"    "RPL15"     "RBMS3"    
 [97] "CLASP2"    "NKTR"      "RPL29"     "ERC2"      "PTPRG"     "CADPS"    
[103] "MAGI1"     "FOXP1"     "ROBO2"     "ROBO1"     "CADM2"     "EPHA3"    
[109] "RPL24"     "CBLB"      "BBX"       "ZBTB20"    "LSAMP"     "GSK3B"    
[115] "KALRN"     "STAG1"     "TNIK"      "NLGN1"     "TBL1XR1"   "IGF2BP2"  
[121] "HES1"      "RPL35A"    "SDK1"      "ACTB"      "PHF14"     "THSD7A"   
[127] "HDAC9"     "TRA2A"     "HNRNPA2B1" "ELMO1"     "AUTS2"     "MAGI2"    
[133] "CACNA2D1"  "CDK14"     "AKAP9"     "KMT2E"     "NRCAM"     "IMMP2L"   
[139] "DOCK4"     "FOXP2"     "KCND2"     "PTPRZ1"    "EXOC4"     "CALD1"    
[145] "PTN"       "TMEM178B"  "CNTNAP2"   "KMT2C"     "DPP6"      "GAPDH"    
[151] "PLEKHA5"   "SOX5"      "ITPR2"     "BICD1"     "KIF21A"    "NELL2"    
[157] "TUBA1A"    "MYL6"      "SRGAP1"    "GRIP1"     "CPSF6"     "NAP1L1"   
[163] "NAV3"      "SYT1"      "PPFIA2"    "TMTC2"     "ANKS1B"    "CHST11"   
[169] "RFX4"      "RPL6"      "MED13L"    "RPLP0"     "RPLP2"     "RPL27A"   
[175] "SBF2"      "TEAD1"     "SOX6"      "RPS13"     "NAV2"      "LUZP2"    
[181] "MPPED2"    "LRRC4C"    "FTH1"      "RTN3"      "FAU"       "MALAT1"   
[187] "TENM4"     "DLG2"      "PICALM"    "FAT3"      "MAML2"     "CNTN5"    
[193] "GRIA4"     "RDX"       "NCAM1"     "CADM1"     "RPS25"     "KIRREL3"  
[199] "NTM"       "OPCML"     "SLIT2"     "PCDH7"     "RPL9"      "LIMCH1"   
[205] "FRYL"      "IGFBP7"    "ADGRL3"    "EPHA5"     "ANKRD17"   "HNRNPDL"  
[211] "WDFY3"     "MAPK10"    "SPARCL1"   "SPP1"      "CCSER1"    "GRID2"    
[217] "UNC5C"     "TSPAN5"    "PPP3CA"    "RPL34"     "ANK2"      "CAMK2D"   
[223] "MAML3"     "DCLK2"     "RPS3A"     "TRIM2"     "GRIA2"     "RAPGEF2"  
[229] "FSTL5"     "CPE"       "GPM6A"     "TENM3"     "SORBS2"    "WSB1"     
[235] "RPL23A"    "NF1"       "RPL23"     "RPL19"     "LUC7L3"    "SPAG9"    
[241] "MSI2"      "TANC2"     "DDX5"      "CEP112"    "BPTF"      "RPL38"    
[247] "ACTG1"     "MYT1L"     "RPS7"      "KIDINS220" "PUM2"      "NCOA1"    
[253] "BIRC6"     "SLC8A1"    "FBXO11"    "NRXN1"     "SPTBN1"    "RTN4"     
[259] "RPS27A"    "CCDC88A"   "LINC01122" "BCL11A"    "USP34"     "XPO1"     
[265] "MEIS1"     "ZNF638"    "EXOC6B"    "LRRTM4"    "CTNNA2"    "TMSB10"   
[271] "AFF3"      "RPL31"     "MAP4K4"    "DPP10"     "CLASP1"    "CNTNAP5"  
[277] "NCKAP5"    "LRP1B"     "ZEB2"      "MBD5"      "FMNL2"     "BAZ2B"    
[283] "KCNH7"     "SCN3A"     "CSRNP3"    "CERS6"     "ZNF804A"   "PGAP1"    
[289] "BMPR2"     "PARD3B"    "MAP2"      "ERBB4"     "FN1"       "RPL37A"   
[295] "PTMA"      "AGAP1"     "HBA2"      "HBA1"      "RPS2"      "RBFOX1"   
[301] "RPS15A"    "TNRC6A"    "TOX3"      "CHD9"      "ZFHX3"     "WWOX"     
[307] "RPL13"

In [56]:
target_genes_avaliable %>% tail(1)

$chunk_51
  [1] "ATXN3L"     "DUSP21"     "SPACA5B"    "XAGE1A"     "SPANXN5"   
  [6] "PAGE5"      "PAGE3"      "CXorf65"    "TEX13A"     "SERPINA7"  
 [11] "TEX13B"     "GPR119"     "MAGEA1"     "DEFB126"    "TMEM239"   
 [16] "CST4"       "WFDC12"     "WFDC10A"    "PRAMEF1"    "PRAMEF6"   
 [21] "GUCA2B"     "KCNA10"     "TCHHL1"     "LCE3C"      "LCE2D"     
 [26] "C1orf68"    "LCE1F"      "IVL"        "SPRR2D"     "SPRR2F"    
 [31] "LELP1"      "PKLR"       "OR10T2"     "OR6K6"      "APCS"      
 [36] "LINC01352"  "BECN2"      "OR2G3"      "OR11L1"     "LINC01556" 
 [41] "OR5V1"      "OR2H1"      "TRIM15"     "LINC00243"  "MUC22"     
 [46] "CFB"        "LINCMD1"    "DPPA5"      "TAAR6"      "AC123023.1"
 [51] "RTP3"       "OR5H14"     "OR5K2"      "PRR23C"     "LINC01323" 
 [56] "AC106706.1" "AC020743.3" "CYP3A7"     "OR9A4"      "CTAGE4"    
 [61] "LALBA"      "LACRT"      "DCD"        "OR10A7"     "MYF5"      
 [66] "SCGB1C1"    "OR52E2"     "OR52N1"     "OR52W1"     "LINC00294" 
 [71] "MIR670HG"   "OR4C12"     "OR4C11"     "OR4P4"      "OR4S2"     
 [76] "OR5D14"     "OR5B21"     "AP000439.3" "OR2AT4"     "TRIM49"    
 [81] "OR10G6"     "OR10G4"     "USP17L10"   "FGFBP1"     "HTN1"      
 [86] "ODAM"       "MTNR1A"     "TLCD2"      "KRT39"      "KRTAP4-7"  
 [91] "KRTAP4-2"   "KRT31"      "KRT38"      "CSH1"       "GALR2"     
 [96] "AC012506.1" "IL36A"      "RPRM"       "HOXD12"     "AC107079.1"
[101] "OR6B3"      "PRM2"       "DEFA6"      "SPAG11B"    "SPAG11A"   
[106] "C8orf49"    "ADAM2"      "SLURP1"     "CYP11B1"    "TIGD5"     
[111] "OR7E24"     "OR10H2"     "ERVK-28"    "FAM138C"    "IFNA16"    
[116] "IFNA17"     "LINC01400"  "SPATA31A6"  "OR13C8"     "OR1L3"     
[121] "OR1K1"      "LINC00421"  "LINC01079"  "LINC01066"  "LINC01078" 
[126] "LINC00443"  "LINC00396"  "OR4K13"     "EDDM3A"     "OR10G2"    
[131] "SERPINA4"   "C14orf180"  "FAM30A"     "TAS2R1"     "ACTBL2"    
[136] "LINC01385"  "FTMT"       "ECSCR"      "OR2Y1"      "ZNF280A"   
[141] "GGTLC2"     "APOL5"      "CALML5"     "NPY4R"      "GDF2"      
[146] "LINC01519"  "TTTY2B"     "TTTY13"     "PRY"        "LRRC30"    
[151] "GOLGA6L22"  "GOLGA8Q"    "OR4F6"      "AP000431.2" "AP000474.1"
[156] "AP001595.1" "KRTAP13-1"  "KRTAP19-3"  "KRTAP19-6"  "KRTAP20-4" 
[161] "KRTAP20-2"  "KRTAP19-8"  "AP001056.1" "KRTAP10-2"  "LINC00165"

In [58]:
# length of each chunk:

target_genes_avaliable %>% lapply(length) %>% unlist() %>% print()

 chunk_1  chunk_2  chunk_3  chunk_4  chunk_5  chunk_6  chunk_7  chunk_8 
     488      489      482      474      476      472      477      478 
 chunk_9 chunk_10 chunk_11 chunk_12 chunk_13 chunk_14 chunk_15 chunk_16 
     477      475      475      475      473      469      470      477 
chunk_17 chunk_18 chunk_19 chunk_20 chunk_21 chunk_22 chunk_23 chunk_24 
     470      458      460      451      457      458      453      442 
chunk_25 chunk_26 chunk_27 chunk_28 chunk_29 chunk_30 chunk_31 chunk_32 
     429      414      399      394      396      382      348      341 
chunk_33 chunk_34 chunk_35 chunk_36 chunk_37 chunk_38 chunk_39 chunk_40 
     313      303      292      291      287      279      256      261 
chunk_41 chunk_42 chunk_43 chunk_44 chunk_45 chunk_46 chunk_47 chunk_48 
     262      259      250      244      234      222      214      242 
chunk_49 chunk_50 chunk_51 
     247      224      165 


In [59]:
# sum of thgem:

target_genes_avaliable %>% lapply(length) %>% unlist() %>% sum()

[1] 19224

In [61]:
# Some of them will be duplicates because of overlapping genes between chunks:

target_genes_avaliable %>% unlist() %>% duplicated() %>% table()

.
FALSE  TRUE 
19042   182 

In [62]:
# Now, when I check back the number of annotation info available genes out of
# all genes in the seurat object, it must be identical with the uniqoe genes
# found above:

# Calculate the number of genes in the RNA assay that are available in the annotation object:

intersect(gene_annot$gene_name, rownames(mannens_et_al_w_eGRNs_SLURM_c1@data)) %>% length()

[1] 19042

In [85]:
# Compare two gene sets:

gene_set_1 <- target_genes_avaliable %>% unlist() %>% unname()

gene_set_2 <- intersect(gene_annot$gene_name, rownames(mannens_et_al_w_eGRNs_SLURM_c1@data))

cat("set 1: ", length(gene_set_1), "\n")

cat("set 2: ", length(gene_set_2), "\n\n")

table(overlap  =gene_set_1 %in% gene_set_2)

set 1:  19224 
set 2:  19042 



overlap
 TRUE 
19224 

=========================================

They both contain exactly same unique genes:

table(gene_set_2 %in% gene_set_1)

 TRUE 
19042

=========================================

They are same. Now, I will calculate the number of overlapping genes\
between two consecutive Pando-selected target gene sets (so called features):

In [86]:
overlapping_gene_num <- vector(mode = "list", length = length(target_genes_avaliable)-1)

for(i in 2:length(target_genes_avaliable)){

    overlapping_gene_num[[i]] <- 
        intersect(target_genes_avaliable[[i-1]], target_genes_avaliable[[i]]) %>% length()

    names(overlapping_gene_num[[i]]) <- paste0('chunk_',i-1, '_vs_chunk_', i)
    
    }

In [87]:
overlapping_gene_num %>% print()

[[1]]
NULL

[[2]]
chunk_1_vs_chunk_2 
                 5 

[[3]]
chunk_2_vs_chunk_3 
                 5 

[[4]]
chunk_3_vs_chunk_4 
                 5 

[[5]]
chunk_4_vs_chunk_5 
                 4 

[[6]]
chunk_5_vs_chunk_6 
                 5 

[[7]]
chunk_6_vs_chunk_7 
                 5 

[[8]]
chunk_7_vs_chunk_8 
                 4 

[[9]]
chunk_8_vs_chunk_9 
                 5 

[[10]]
chunk_9_vs_chunk_10 
                  5 

[[11]]
chunk_10_vs_chunk_11 
                   5 

[[12]]
chunk_11_vs_chunk_12 
                   5 

[[13]]
chunk_12_vs_chunk_13 
                   5 

[[14]]
chunk_13_vs_chunk_14 
                   3 

[[15]]
chunk_14_vs_chunk_15 
                   5 

[[16]]
chunk_15_vs_chunk_16 
                   4 

[[17]]
chunk_16_vs_chunk_17 
                   5 

[[18]]
chunk_17_vs_chunk_18 
                   4 

[[19]]
chunk_18_vs_chunk_19 
                   5 

[[20]]
chunk_19_vs_chunk_20 
                   4 

[[21]]
chunk_20_vs_chunk_21 
             

In [88]:
sapply(overlapping_gene_num, function(x) x > 0) %>% unlist() %>% table()

.
FALSE  TRUE 
    4    46 

In [89]:
# Except four of the eGRN objects, all contains some number of overlapping genes.

# I will compare the coefficients and number of variables predicted for those 
# overlapping genes in the combined Pando eGRN object.

===================================================================================================

I used the following commands in a tmux session to combine coefficient data.frames, fit data.frames\
and features vectors from all eGRN objects:

In [90]:
# getwd()

# here::here()

# # load the R environment with the necessary packages such as Epiregulon:

# my_epiregulon_lib <- here::here("renv", "library/linux-rhel-9.4/R-4.4/x86_64-unknown-linux-gnu")

# .libPaths(new = my_epiregulon_lib, include.site = FALSE)

# .libPaths()

# library(magrittr)

# here::here('r_objects') |> list.dirs() %>% basename()

# # view eGRN objects in windows-style order:

# here::here('r_objects', 'pando_eGRN_chunks_final') |> list.files() %>% gtools::mixedsort() %>% print()

# library(tidyverse)

# library(Seurat)

# library(Signac)

# library(Pando)

# # Import first eGRN object to access Seurat object:

# mannens_et_al_w_eGRNs_SLURM_c1 <- 
#     readRDS(here::here('r_objects', 'pando_eGRN_chunks_final', 'mannens_et_al_w_eGRNs_SLURM_c1.RDS'))

# coef_DF <- coef(mannens_et_al_w_eGRNs_SLURM_c1) 

# fit_DF <-  mannens_et_al_w_eGRNs_SLURM_c1@grn@networks$glm_network@fit

# coef_DF %>% head()

# fit_DF %>% head()

# features_vector <- NetworkFeatures(mannens_et_al_w_eGRNs_SLURM_c1)

# file_prefix <- "mannens_et_al_w_eGRNs_SLURM_c"

# for(i in c(2:51)) {

#  message(glue::glue("reading chunk ",{i}))
    
#  chunk_obj <- readRDS(here::here('r_objects', 'pando_eGRN_chunks_final', paste0(file_prefix, i, ".RDS")))

#  coef_DF <- rbind(coef_DF, coef(chunk_obj))

#  fit_DF <- rbind(fit_DF, chunk_obj@grn@networks$glm_network@fit)

#  features_vector <- c(features_vector, NetworkFeatures(chunk_obj))

#  rm(chunk_obj)
                      
#  message(glue::glue("added chunk ",{i}))

# }    

# coef_DF %>% saveRDS(here::here('r_objects', 'coef_DF_combined.RDS'))

# fit_DF %>% saveRDS(here::here('r_objects', 'fit_DF_combined.RDS'))

# features_vector %>% saveRDS(here::here('r_objects', 'features_vector_combined.RDS'))

# ## end ##

In [91]:
ls()

[1] "gene_annot"                     "gene_chunks"                   
 [3] "gene_set_1"                     "gene_set_2"                    
 [5] "i"                              "mannens_et_al_w_eGRNs_SLURM_c1"
 [7] "my_epiregulon_lib"              "overlapping_gene_num"          
 [9] "process_pando_networks"         "retrieve_in_chunks"            
[11] "target_genes_avaliable"

In [92]:
coef_DF <- readRDS(here::here('r_objects', 'coef_DF_combined.RDS'))

In [93]:
fit_DF <- readRDS(here::here('r_objects', 'fit_DF_combined.RDS'))

In [94]:
features_vector <- readRDS(here::here('r_objects', 'features_vector_combined.RDS'))

In [100]:
coef_DF$target %>% unique() %>% length()

[1] 6893

In [101]:
fit_DF$target %>% unique() %>% length()

[1] 7021

In [102]:
features_vector  %>% unique() %>% length()

[1] 19042

In [103]:
mannens_et_al_w_eGRNs_SLURM_c1 %>% coef() %>% pull(target) %>% unique() %>% length()

[1] 480

In [104]:
mannens_et_al_w_eGRNs_SLURM_c1 %>% NetworkFeatures() %>% unique() %>% length()

[1] 488

In [105]:
# number of features initially selected by Pando is not same
# with the unique target genes in coefficient table.

In [106]:
# Import the last eGRN object to test this:

mannens_et_al_w_eGRNs_SLURM_c51 <- 
    readRDS(here::here('r_objects', 'pando_eGRN_chunks_final', 'mannens_et_al_w_eGRNs_SLURM_c51.RDS'))

In [107]:
mannens_et_al_w_eGRNs_SLURM_c51 %>% coef() %>% pull(target) %>% unique() %>% length()

[1] 9

In [108]:
mannens_et_al_w_eGRNs_SLURM_c51 %>% NetworkFeatures() %>% unique() %>% length()

[1] 165

In [109]:
mannens_et_al_w_eGRNs_SLURM_c51 %>% coef()

tf,target,region,term,estimate,std_err,statistic,pval,padj,corr
<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
NR1H4,SPANXN5,chrX-53262222-53262552,chrX_53262222_53262552:NR1H4,-2.249553e-05,0.0050033730,-4.496074e-03,9.964127e-01,9.996479e-01,0.1138566
NR1H4,SPANXN5,chrX-53716545-53716895,NR1H4:chrX_53716545_53716895,-4.625607e-06,0.0010288102,-4.496074e-03,9.964127e-01,9.996479e-01,0.1138566
CDX1,DEFB126,chr20-237500-237847,chr20_237500_237847:CDX1,-1.079972e-05,0.0024020326,-4.496074e-03,9.964127e-01,9.996479e-01,0.1245630
CDX1,DEFB126,chr20-953832-954047,CDX1:chr20_953832_954047,-1.151419e-05,0.0025609440,-4.496074e-03,9.964127e-01,9.996479e-01,0.1245630
DMRT1,AC020743.3,chr7-50148318-50148502,DMRT1:chr7_50148318_50148502,-7.224394e-06,0.0011383185,-6.346549e-03,9.949362e-01,9.996479e-01,0.1169837
DMRT1,AC020743.3,chr7-50858141-50858268,DMRT1:chr7_50858141_50858268,-3.180913e-06,0.0007074360,-4.496396e-03,9.964124e-01,9.996479e-01,0.1169837
DMRT1,AC020743.3,chr7-51161051-51161397,DMRT1:chr7_51161051_51161397,-1.523857e-05,0.0021194212,-7.189969e-03,9.942633e-01,9.996479e-01,0.1169837
DMRT1,AC020743.3,chr7-51357206-51357469,DMRT1:chr7_51357206_51357469,-2.561965e-06,0.0005697818,-4.496396e-03,9.964124e-01,9.996479e-01,0.1169837
VDR,OR10G6,chr11-123033900-123034098,chr11_123033900_123034098:VDR,-2.285414e-06,0.0014259895,-1.602686e-03,9.987212e-01,9.996479e-01,0.1021516


In [110]:
# There are so less number of target genes.

R crashed. I could not print sessioninfo().